In [30]:
import numpy as np
from collections import defaultdict, Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [31]:
dataset = load_dataset("lhoestq/conll2003")

features = dataset["train"].features

tag_names = [
    "O", "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

In [32]:
def extract_data(split):
    sentences = []
    tags = []

    for example in split:
        sentences.append(example["tokens"])
        tags.append([tag_names[tag] for tag in example["ner_tags"]])

    return sentences, tags

In [33]:
train_sentences, train_tags = extract_data(dataset["train"])
test_sentences, test_tags = extract_data(dataset["test"])

In [34]:
class HMM_NER:
    def __init__(self):
        self.transition_probs = defaultdict(lambda: defaultdict(float))
        self.emission_probs = defaultdict(lambda: defaultdict(float))
        self.tag_counts = Counter()
        self.tags = set()
        self.vocab = set()

    def train(self, sentences, tags):
        transition_counts = defaultdict(lambda: defaultdict(int))
        emission_counts = defaultdict(lambda: defaultdict(int))

        for sent, tag_seq in zip(sentences, tags):
            prev_tag = "<START>"

            for word, tag in zip(sent, tag_seq):
                self.vocab.add(word)
                self.tags.add(tag)

                transition_counts[prev_tag][tag] += 1
                emission_counts[tag][word] += 1
                self.tag_counts[tag] += 1

                prev_tag = tag

            transition_counts[prev_tag]["<END>"] += 1

        # Transition probabilities (Laplace smoothing)
        for prev_tag in transition_counts:
            total = sum(transition_counts[prev_tag].values())
            for tag in transition_counts[prev_tag]:
                self.transition_probs[prev_tag][tag] = (
                    transition_counts[prev_tag][tag] + 1
                ) / (total + len(self.tags))

        # Emission probabilities (Laplace smoothing)
        for tag in emission_counts:
            total = sum(emission_counts[tag].values())
            for word in emission_counts[tag]:
                self.emission_probs[tag][word] = (
                    emission_counts[tag][word] + 1
                ) / (total + len(self.vocab))

    def viterbi(self, sentence):
        V = [{}]
        path = {}

        # Initialization
        for tag in self.tags:
            trans = self.transition_probs["<START>"].get(tag, 1e-6)
            emit = self.emission_probs[tag].get(sentence[0], 1e-6)
            V[0][tag] = np.log(trans) + np.log(emit)
            path[tag] = [tag]

        # Recursion
        for t in range(1, len(sentence)):
            V.append({})
            new_path = {}

            for curr_tag in self.tags:
                max_prob = float("-inf")
                best_prev = None

                for prev_tag in self.tags:
                    prob = (
                        V[t - 1][prev_tag]
                        + np.log(self.transition_probs[prev_tag].get(curr_tag, 1e-6))
                        + np.log(self.emission_probs[curr_tag].get(sentence[t], 1e-6))
                    )

                    if prob > max_prob:
                        max_prob = prob
                        best_prev = prev_tag

                V[t][curr_tag] = max_prob
                new_path[curr_tag] = path[best_prev] + [curr_tag]

            path = new_path

        # Termination
        max_prob = float("-inf")
        best_tag = None

        for tag in self.tags:
            prob = V[-1][tag] + np.log(self.transition_probs[tag].get("<END>", 1e-6))
            if prob > max_prob:
                max_prob = prob
                best_tag = tag

        return path[best_tag]

    def predict(self, sentences):
        return [self.viterbi(sent) for sent in sentences]

In [35]:
def evaluate(true_tags, pred_tags):
    y_true = [tag for sent in true_tags for tag in sent]
    y_pred = [tag for sent in pred_tags for tag in sent]

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='weighted', zero_division=0))
    print("Recall:", recall_score(y_true, y_pred, average='weighted', zero_division=0))
    print("F1 Score:", f1_score(y_true, y_pred, average='weighted', zero_division=0))

In [36]:
hmm = HMM_NER()
hmm.train(train_sentences, train_tags)

predictions = hmm.predict(test_sentences[:200])  # limit for speed

evaluate(test_tags[:200], predictions)

Accuracy: 0.9048543689320389
Precision: 0.911205579542615
Recall: 0.9048543689320389
F1 Score: 0.8917019355568835
